In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))
import numpy as np
import pandas as pd
import random
from scripts.cleaning_gpt_data import get_raw_data, process_txt_dataset, extract_personal_info, all_questionnaires, process_associations, find_missing_cue_words
from scripts.mas_it_df_splitting import low_high_anxiety_division

In [2]:
pd.set_option('display.max_colwidth', None)

## Data extraction and cleaning

In [3]:
folder_gpt_oss = f'/Users/emma/projects/cognoscolab/rassereno/data/gpt_oss_students'

raw_data_gpt_oss = get_raw_data(folder_gpt_oss)
cleaned_data_gpt_oss = process_txt_dataset(raw_data_gpt_oss, 'gpt_oss')
personal_data_gpt_oss = extract_personal_info(cleaned_data_gpt_oss)
all_questionnaires_gpt_oss, mas_it_gpt_oss = all_questionnaires(personal_data_gpt_oss)
association_valence_gpt_oss = process_associations(cleaned_data_gpt_oss)
find_missing_cue_words(association_valence_gpt_oss, personal_data_gpt_oss)

Number of generated files: 1000
Number of cue words: 41
Number of participants: 999
⚠️ Missing cue words found
      participant_id     cue_word
23       gpt_oss_001  neurosienze
64       gpt_oss_002  neurosienze
72       gpt_oss_002       sapere
105      gpt_oss_003  neurosienze
113      gpt_oss_003       sapere
...              ...          ...
40954    gpt_oss_305     sviluppo
40955    gpt_oss_305    terapeuta
40956    gpt_oss_305         test
40957    gpt_oss_305   universita
40958    gpt_oss_305  valutazione

[1717 rows x 2 columns]
ansia : 986
attaccamento : 996
benessere : 995
biologia : 997
cognizione : 997
comportamento : 994
creativita : 989
curiosita : 968
divertimento : 989
emozione : 996
equazione : 992
esame : 996
fisica : 995
informatica : 999
innovazione : 983
insegnante : 988
laurea : 990
lavoro : 986
matematica : 996
media : 987
mente : 998
modello : 985
neuroscienze : 994
neurosienze : 3
passione : 989
personalita : 997
professore : 981
prova : 988
psicologia : 998
p

In [4]:
# drops the participants with more than 3 missing MAS-IT scores
to_drop = mas_it_gpt_oss['participant_id'][mas_it_gpt_oss.isnull().sum(axis=1) > 3]
all_questionnaires_gpt_oss = all_questionnaires_gpt_oss[~all_questionnaires_gpt_oss['participant_id'].isin(to_drop)]

In [5]:
# check the participants we dropped with the previous code
for i in to_drop:
    print(cleaned_data_gpt_oss[cleaned_data_gpt_oss['participant_id'] == i][['file_path', 'mas_it']])

                                                                            file_path  \
49  /Users/emma/projects/cognoscolab/rassereno/data/gpt_oss_students/studente_483.txt   

                                                                                           mas_it  
49  [nan, 5, nan, 4, 3, nan, nan, 5, 2, nan, 4, 3, nan, 5, 4, 3, nan, nan, nan, nan, 5, nan, nan]  
                                                                            file_path  \
67  /Users/emma/projects/cognoscolab/rassereno/data/gpt_oss_students/studente_132.txt   

                                                                                                           mas_it  
67  [nan, nan, nan, 5, 4, nan, nan, nan, nan, nan, 5, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]  
                                                                           file_path  \
75  /Users/emma/projects/cognoscolab/rassereno/data/gpt_oss_students/studente_96.txt   

   mas_it  
75   None  
         

## Randomly generate samples and split of GPT-OSS

In [6]:
# from the df with the dropped participants, get the psychology, pysics and high school participants
fisica_participants_gpt = all_questionnaires_gpt_oss['participant_id'][all_questionnaires_gpt_oss['faculty'].str.startswith('uni_fisica')]
psychology_participants_gpt = all_questionnaires_gpt_oss['participant_id'][all_questionnaires_gpt_oss['faculty'].str.startswith('uni_psicologia')]
liceo_participants_gpt = all_questionnaires_gpt_oss['participant_id'][all_questionnaires_gpt_oss['faculty'].str.startswith('liceo')]

print('Fisica:', len(fisica_participants_gpt))
print('Psicologia:', len(psychology_participants_gpt))
print('Liceo:', len(liceo_participants_gpt))

Fisica: 358
Psicologia: 316
Liceo: 258


In [7]:
path_rassereno = '/Users/emma/projects/cognoscolab/rassereno/'

In [8]:
# counts the number of human students, that we'll have to match with GPT-OSS students
n_all_psy_students = len(pd.read_csv(f'{path_rassereno}/out/formatting/all_psychology_students/mas_it_all_psy_students.csv'))
n_liceo_students = len(pd.read_csv(f'{path_rassereno}/out/formatting/liceo_2025/mas_it_liceo_2025.csv'))
n_physics_students = len(pd.read_csv(f'{path_rassereno}/out/formatting/physics_2025/mas_it_fisica_2025.csv'))

In [9]:
# randomly takes the number of required students (to match the human ones) from each category
random.seed(42) # for reproducibility
sample_psy_students = random.sample(list(psychology_participants_gpt), n_all_psy_students)
sample_physics_students = random.sample(list(fisica_participants_gpt), n_physics_students)
sample_liceo_students = random.sample(list(liceo_participants_gpt), n_liceo_students)

### Final dfs

In [22]:
association_valence_physics = association_valence_gpt_oss[association_valence_gpt_oss['participant_id'].isin(sample_physics_students)]
association_valence_physics.to_csv(f'{path_rassereno}/out/formatting/gpt_oss/associations_valence_physics.csv', index=False)
mas_it_gpt_physics = mas_it_gpt_oss[mas_it_gpt_oss['participant_id'].isin(sample_physics_students)]
mas_it_gpt_physics.to_csv(f'{path_rassereno}/out/formatting/gpt_oss/mas_it_physics.csv', index=False)

In [20]:
# split the gpt_oss dataframes according to the sampled psychology participants
mas_it_gpt_psychology = mas_it_gpt_oss[mas_it_gpt_oss['participant_id'].isin(sample_psy_students)]
mas_it_gpt_psychology.to_csv(f'{path_rassereno}/out/formatting/gpt_oss/mas_it_psychology.csv', index=False)
association_valence_psychology = association_valence_gpt_oss[association_valence_gpt_oss['participant_id'].isin(sample_psy_students)]
association_valence_psychology.to_csv(f'{path_rassereno}/out/formatting/gpt_oss/associations_valence_psychology.csv', index=False)
# low and high anxiety division of the df and saving to csv
low_high_anxiety_division(mas_it_gpt_psychology, association_valence_psychology, 'psychology_gpt_oss', path_rassereno);

Results for psychology_gpt_oss sample 

total median of the summed scores: 80.0
number of total students: 316
number of students with high anxiety: 143
number of students with low anxiety: 157
number of students to exclude: 16


In [ ]:
# split the gpt_oss dataframes according to the sampled high school participants
association_valence_liceo = association_valence_gpt_oss[association_valence_gpt_oss['participant_id'].isin(sample_liceo_students)]
association_valence_liceo.to_csv(f'{path_rassereno}/out/formatting/gpt_oss/associations_valence_liceo.csv', index=False)

mas_it_gpt_liceo = mas_it_gpt_oss[mas_it_gpt_oss['participant_id'].isin(sample_liceo_students)]
mas_it_gpt_liceo.to_csv(f'{path_rassereno}/out/formatting/gpt_oss/mas_it_liceo.csv', index=False)

low_high_anxiety_division(mas_it_gpt_liceo, association_valence_liceo, 'high_school_gpt_oss', path_rassereno);

Results for high_school_gpt_oss sample 

total median of the summed scores: 81.0
number of total students: 62
number of students with high anxiety: 28
number of students with low anxiety: 29
number of students to exclude: 5
